# ID extraction: passports and ID cards

Two independent reads of every ID:

- **MRZ** (machine-readable zone): OCR the bottom lines, parse them, verify the check digits.
- **Printed zone**: look for each MRZ value where it is printed on the card ("BLONDEL", "31.12.1985").
  This works on any design because it doesn't need to know where the labels are.

A field is green only when both agree. A check digit alone is not enough (see the G/6 case below).

In [ ]:
from pathlib import Path
import os, sys
if Path(os.getcwd()).name == "notebooks":
    os.chdir("..")  # run from the repo root so the pipeline imports work
sys.path.insert(0, os.getcwd())
import json
import pandas as pd
from config import DATASET
from extract.id_document import read_id_document
from extract.mrz import check_digit

labels = json.loads((DATASET / "labels.json").read_text())

def id_files(l):
    return [DATASET / l["investor_id"] / name for name, kind in l["files"].items() if kind != "subscription"]

## One passport, step by step

In [ ]:
first = next(l for l in labels if l["id_kind"] == "passport")
raw = read_id_document(id_files(first))
print("Format:", raw["format"])
pd.DataFrame({"MRZ": raw["mrz"], "check digit": raw["mrz_checks"], "found printed as": raw["printed"]})

## Why a check digit alone can't be trusted

ICAO check digits weight characters 7-3-1 and keep the total mod 10. The letter `G` is worth 16 and
the digit `6` is worth 6. They differ by 10, so swapping one for the other never changes the check digit.

In [ ]:
print(check_digit("AB1G23456"), check_digit("AB1623456"))  # same check digit, different numbers

## How often the MRZ is read correctly, by ID design

In [ ]:
rows = []
for l in labels:
    raw = read_id_document(id_files(l))
    rows.append({"design": l["id_template"], "mrz_found": raw["format"] is not None,
                 "all_checks_pass": bool(raw["mrz_checks"]) and all(raw["mrz_checks"].values()),
                 "number_right": raw["mrz"].get("document_number") == l["id"]["document_number"]})
results = pd.DataFrame(rows)
results.groupby("design").mean(numeric_only=True).style.format("{:.0%}")